In [7]:
import pandas as pd
import numpy as np

# P&G historical financial statements (USD millions)
historical = pd.DataFrame({
    "Revenue": [84039, 84284, 87032],
    "EBIT": [18545, 20451, 19748],
    "Income Taxes": [3787, 4102, 4233],
    "Earnings Before Taxes": [18761, 20167, 20377],
    "D&A": [2896, 2847, 3160],
    "CapEx": [3322, 3773, 4409],
    "NWC": [-11208, -10709, -11131]
}, index=[2024, 2025, 2026])

historical["Revenue Growth"] = historical["Revenue"].pct_change()
historical["Operating Margin"] = historical["EBIT"] / historical["Revenue"]
historical["Effective Tax Rate"] = historical["Income Taxes"] / historical["Earnings Before Taxes"]

print(historical.round(4))

      Revenue   EBIT  Income Taxes  Earnings Before Taxes   D&A  CapEx    NWC  \
2024    84039  18545          3787                  18761  2896   3322 -11208   
2025    84284  20451          4102                  20167  2847   3773 -10709   
2026    87032  19748          4233                  20377  3160   4409 -11131   

      Revenue Growth  Operating Margin  Effective Tax Rate  
2024             NaN            0.2207              0.2019  
2025          0.0029            0.2426              0.2034  
2026          0.0326            0.2269              0.2077  


In [8]:
forecast = pd.DataFrame(index=range(2027, 2032))

# Assumptions from our Google Sheets DCF Forecast
forecast["Revenue Growth"] = [0.02, 0.025, 0.027, 0.03, 0.03]
forecast["Operating Margin"] = [0.23, 0.232, 0.234, 0.235, 0.235]
forecast["Tax Rate"] = [0.21] * 5
forecast["D&A / Revenue"] = [0.035] * 5
forecast["CapEx / Revenue"] = [0.05, 0.05, 0.048, 0.047, 0.045]

# Calculate future revenue
previous_revenue = historical.loc[2026, "Revenue"]

for year in forecast.index:
    growth = forecast.loc[year, "Revenue Growth"]
    previous_revenue = previous_revenue * (1 + growth)
    forecast.loc[year, "Revenue"] = previous_revenue

print(forecast.round(4))

      Revenue Growth  Operating Margin  Tax Rate  D&A / Revenue  \
2027           0.020             0.230      0.21          0.035   
2028           0.025             0.232      0.21          0.035   
2029           0.027             0.234      0.21          0.035   
2030           0.030             0.235      0.21          0.035   
2031           0.030             0.235      0.21          0.035   

      CapEx / Revenue     Revenue  
2027            0.050  88772.6400  
2028            0.050  90991.9560  
2029            0.048  93448.7388  
2030            0.047  96252.2010  
2031            0.045  99139.7670  


In [9]:
forecast["EBIT"] = forecast["Revenue"] * forecast["Operating Margin"]
forecast["EBIT After Tax"] = forecast["EBIT"] * (1 - forecast["Tax Rate"])
forecast["D&A"] = forecast["Revenue"] * forecast["D&A / Revenue"]
forecast["CapEx"] = forecast["Revenue"] * forecast["CapEx / Revenue"]

# NWC assumption based on 2026
nwc_ratio = historical.loc[2026, "NWC"] / historical.loc[2026, "Revenue"]

forecast["NWC"] = forecast["Revenue"] * nwc_ratio

previous_nwc = historical.loc[2026, "NWC"]

for year in forecast.index:
    forecast.loc[year, "Change in NWC"] = forecast.loc[year, "NWC"] - previous_nwc
    previous_nwc = forecast.loc[year, "NWC"]

forecast["FCFF"] = (
    forecast["EBIT After Tax"]
    + forecast["D&A"]
    - forecast["CapEx"]
    - forecast["Change in NWC"]
)

print(forecast[[
    "Revenue", "EBIT", "D&A", "CapEx",
    "NWC", "Change in NWC", "FCFF"
]].round(2))

       Revenue      EBIT      D&A    CapEx       NWC  Change in NWC      FCFF
2027  88772.64  20417.71  3107.04  4438.63 -11353.62        -222.62  15021.02
2028  90991.96  21110.13  3184.72  4549.60 -11637.46        -283.84  15595.97
2029  93448.74  21867.00  3270.71  4485.54 -11951.67        -314.21  16374.31
2030  96252.20  22619.27  3368.83  4523.85 -12310.22        -358.55  17072.74
2031  99139.77  23297.85  3469.89  4461.29 -12679.53        -369.31  17783.21


In [10]:

# WACC assumptions
risk_free_rate = 0.0510
beta = 0.3843
equity_risk_premium = 0.0420
cost_of_debt = 0.0567
tax_rate = 0.21

share_price = 159.35
common_shares = 2323.859
diluted_shares = 2412.659

short_term_debt = 11296
long_term_debt = 22842
cash = 9942
noncontrolling_interest = 230

# WACC calculation
cost_of_equity = risk_free_rate + beta * equity_risk_premium

total_debt = short_term_debt + long_term_debt
market_cap = share_price * common_shares

equity_weight = market_cap / (market_cap + total_debt)
debt_weight = total_debt / (market_cap + total_debt)

wacc = (equity_weight * cost_of_equity
        + debt_weight * cost_of_debt * (1 - tax_rate))

# DCF calculation
terminal_growth = 0.025

forecast["Year (t)"] = np.arange(1, 6)
forecast["PV of FCFF"] = (
    forecast["FCFF"] / (1 + wacc) ** forecast["Year (t)"]
)

pv_forecast_fcff = forecast["PV of FCFF"].sum()

terminal_value = (
    forecast.loc[2031, "FCFF"] * (1 + terminal_growth)
    / (wacc - terminal_growth)
)

pv_terminal_value = terminal_value / (1 + wacc) ** 5
enterprise_value = pv_forecast_fcff + pv_terminal_value

net_debt = total_debt - cash
equity_value = enterprise_value - net_debt - noncontrolling_interest

value_per_share = equity_value / diluted_shares

print(f"Cost of Equity: {cost_of_equity:.4%}")
print(f"WACC: {wacc:.4%}")
print(f"Enterprise Value: {enterprise_value:,.2f} million USD")
print(f"Equity Value: {equity_value:,.2f} million USD")
print(f"DCF Value per Share: ${value_per_share:.2f}")


Cost of Equity: 6.7141%
WACC: 6.5254%
Enterprise Value: 397,720.19 million USD
Equity Value: 373,294.19 million USD
DCF Value per Share: $154.72


In [11]:

# Sensitivity Analysis: WACC vs Terminal Growth Rate

wacc_scenarios = wacc + np.array([-0.01, -0.005, 0, 0.005, 0.01])
growth_scenarios = [0.015, 0.02, 0.025, 0.03, 0.035]

sensitivity = pd.DataFrame(
    index=wacc_scenarios,
    columns=growth_scenarios,
    dtype=float
)

for w in wacc_scenarios:
    for g in growth_scenarios:
        # Present value of forecast FCFF
        pv_fcff = sum(
            forecast["FCFF"] / (1 + w) ** forecast["Year (t)"]
        )

        # Terminal value and its present value
        tv = forecast.loc[2031, "FCFF"] * (1 + g) / (w - g)
        pv_tv = tv / (1 + w) ** 5

        # Equity value per share
        ev = pv_fcff + pv_tv
        equity = ev - net_debt - noncontrolling_interest
        sensitivity.loc[w, g] = equity / diluted_shares

sensitivity.index = [f"{w:.2%}" for w in wacc_scenarios]
sensitivity.columns = [f"{g:.2%}" for g in growth_scenarios]

print("Sensitivity Analysis — Value per Share (USD)")
display(sensitivity.round(2))


Sensitivity Analysis — Value per Share (USD)


,1.50%,2.00%,2.50%,3.00%,3.50%
5.53%,160.73,181.67,209.53,248.43,306.54
6.03%,141.68,157.69,178.24,205.58,243.75
6.53%,126.43,139.01,154.72,174.89,201.72
7.03%,113.94,124.05,136.40,151.82,171.62
7.53%,103.52,111.80,121.73,133.86,148.99


In [12]:

from scipy.optimize import brentq

# Calculate DCF value for a constant annual revenue growth rate
def reverse_dcf_value(growth):
    revenues = []
    previous_revenue = historical.loc[2026, "Revenue"]

    for year in forecast.index:
        previous_revenue *= (1 + growth)
        revenues.append(previous_revenue)

    revenues = np.array(revenues)

    ebit = revenues * forecast["Operating Margin"].to_numpy()
    nopat = ebit * (1 - forecast["Tax Rate"].to_numpy())
    da = revenues * forecast["D&A / Revenue"].to_numpy()
    capex = revenues * forecast["CapEx / Revenue"].to_numpy()

    nwc = revenues * nwc_ratio
    changes_nwc = np.diff(
        np.insert(nwc, 0, historical.loc[2026, "NWC"])
    )

    fcff = nopat + da - capex - changes_nwc

    years = np.arange(1, 6)
    pv_fcff = np.sum(fcff / (1 + wacc) ** years)

    tv = fcff[-1] * (1 + terminal_growth) / (wacc - terminal_growth)
    ev = pv_fcff + tv / (1 + wacc) ** 5

    return (ev - net_debt - noncontrolling_interest) / diluted_shares


# Find growth rate that matches observed share price
implied_growth = brentq(
    lambda growth: reverse_dcf_value(growth) - share_price,
    -0.20,
    0.20
)

print(f"Implied Annual Revenue Growth: {implied_growth:.4%}")
print(f"Reverse DCF Share Value: ${reverse_dcf_value(implied_growth):.4f}")
print(f"Market Share Price: ${share_price:.4f}")


Implied Annual Revenue Growth: 3.1942%
Reverse DCF Share Value: $159.3500
Market Share Price: $159.3500
